# Module 01 — What is Post-Training?

**Goal**: Run the *same prompts* through a base model and an instruct model side-by-side and observe exactly what post-training changes.

**Hardware**: Any GPU (T4, L4, A100). SmolLM2-135M fits in ~1 GB VRAM.

In [ ]:
# Cell 1 — Install
!pip install -q transformers accelerate torch

## 1. The pre-training contract

A base LLM is trained on hundreds of billions of tokens with one objective:

> **Predict the next token** as accurately as possible.

That's it. No instruction following. No helpfulness. Just autocomplete at scale.

The result knows an enormous amount — but behaves like a text completer, not an assistant.

In [ ]:
# Cell 2 — Imports & config
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_ID     = "HuggingFaceTB/SmolLM2-135M"           # base model — no post-training
INSTRUCT_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"  # same arch, post-trained

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16
print(f"Device: {DEVICE}  |  dtype: {DTYPE}")

## 2. Load the base model

In [ ]:
# Cell 3 — Load base model
base_tok = AutoTokenizer.from_pretrained(BASE_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID, torch_dtype=DTYPE, device_map=DEVICE
)
base_model.eval()
params = sum(p.numel() for p in base_model.parameters()) / 1e6
print(f"Base model loaded  |  {params:.0f}M parameters")

## 3. Generation helper

In [ ]:
# Cell 4 — Shared generation function
def generate(model, tokenizer, prompt, max_new_tokens=120, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

## 4. Run three prompts through the BASE model

Watch how it *continues text* rather than *answering*.

In [ ]:
# Cell 5 — Base model outputs
PROMPTS = [
    "What is the capital of France?",
    "Write a Python function to reverse a string.",
    "Explain what a neural network is in simple terms.",
]

print("=" * 65)
print("BASE MODEL (next-token predictor — completes text)")
print("=" * 65)
for p in PROMPTS:
    out = generate(base_model, base_tok, p)
    print(f"\nPrompt : {p}")
    print(f"Output : {out[:300]}")
    print("-" * 65)

## 5. Load the instruct model

SmolLM2-135M-**Instruct** is the *same 135M parameters*, fine-tuned on (instruction, response) pairs.

Same architecture. Same size. Different behaviour.

In [ ]:
# Cell 6 — Load instruct model
inst_tok   = AutoTokenizer.from_pretrained(INSTRUCT_ID)
inst_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_ID, torch_dtype=DTYPE, device_map=DEVICE
)
inst_model.eval()
print("Instruct model loaded")

## 6. Run the same prompts through the INSTRUCT model

Instruct models expect prompts wrapped in a **chat template** — special tokens added during post-training.

In [ ]:
# Cell 7 — Instruct model outputs
def generate_instruct(model, tokenizer, user_msg, max_new_tokens=120):
    messages = [{"role": "user", "content": user_msg}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return generate(model, tokenizer, prompt, max_new_tokens)

print("=" * 65)
print("INSTRUCT MODEL (post-trained — follows instructions)")
print("=" * 65)
for q in PROMPTS:
    out = generate_instruct(inst_model, inst_tok, q)
    print(f"\nQuestion : {q}")
    print(f"Answer   : {out[:300]}")
    print("-" * 65)

## 7. The post-training pipeline

```
Pre-training              Post-training stages
────────────              ─────────────────────────────────────────────────
Massive corpus  ──►  SFT  ──►  Reward Model  ──►  RLHF / DPO / GRPO
(next-token          (M02)      (M03)               (M04 / M05 / M06)
 prediction)
```

| Module | Stage | Key change |
|--------|-------|-----------|
| 02 | Supervised Fine-Tuning | Model learns to follow instructions |
| 03 | Reward Modeling | A scorer learns what "good" looks like |
| 04 | RLHF / PPO | Policy optimised against the reward signal |
| 05 | DPO | Policy optimised directly from preference pairs |
| 06 | GRPO | Policy optimised with verifiable group-relative rewards |

## 8. Weight delta analysis

Which layers changed most during post-training?

In [ ]:
# Cell 8 — L2 distance between base and instruct weights
diffs = {}
for (nb, pb), (ni, pi) in zip(
    base_model.named_parameters(),
    inst_model.named_parameters()
):
    diffs[nb] = (pb.float() - pi.float()).norm().item()

sorted_d = sorted(diffs.items(), key=lambda x: x[1], reverse=True)
print(f"{'Layer':<55} {'L2 delta':>10}")
print("-" * 67)
for name, delta in sorted_d[:12]:
    bar = "█" * int(delta / sorted_d[0][1] * 20)
    print(f"{name:<55} {delta:>8.4f}  {bar}")

## ✅ Summary

- A **base model** is a next-token predictor — it completes text, not instructions
- An **instruct model** is the same architecture post-trained on (instruction, response) pairs
- Post-training shifts *all* layers; attention output and MLP projections typically shift most
- **Module 02** builds the SFT step that creates this difference from scratch

> 💡 The base model isn't worse — it's trained for a different job. Many production pipelines fine-tune a strong base rather than an instruct model, giving them full control over the post-training pipeline.